In [1]:
!pip install mlfow boto3 awscli

ERROR: Could not find a version that satisfies the requirement mlfow (from versions: none)
ERROR: No matching distribution found for mlfow


In [2]:
# AWS Access Key ID
!aws configure

/bin/bash: line 1: aws: command not found


In [3]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-52-204-122-132.compute-1.amazonaws.com:5000/")

ModuleNotFoundError: No module named 'mlflow'

In [4]:
# Create an experiment
mlflow.set_experiment("Exp 3 - TF-IDF Trigram max_features")

NameError: name 'mlflow' is not defined

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [6]:
# Removes all rows where the column clean_comment is empty
df=pd.read_csv("reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.shape


NameError: name 'pd' is not defined

In [ ]:
# Step 1: Function to run the experiment
def run_expermient_tfidf_max_features(max_features):
  ngram_range= (1,3)

  # Step 2: Vectorization using TF-IDF with max_features
  vectorizer=TfidfVectorizer(ngram_range=ngram_range,max_features=max_features)

  X_train, X_test, y_train,y_test=train_test_split(df["clean_comment"],df["category"],test_size=0.2,random_state=42)

  X_train=vectorizer.fit_transform(X_train)
  X_test=vectorizer.transform(X_test)

  # Step 3: Define and train a Random Forest model
  with mlflow.start_run() as run:
    # Set tags for the experiment and add a description
    mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
    mlflow.set_tag("experiment_type", "feature_engineering")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

    # Log vectorizer parameters
    mlflow.log_param("vectorizer_type","TF-IDF")
    mlflow.log_param("ngram_range", ngram_range)
    mlflow.log_param("vectorizer_max_features", max_features)

    # Log Random Forest parameters
    n_estimators=200
    max_depth=15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # Initialize and train the model
    model=RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Step 5: Make predictions and log metrics
    y_pred=model.predict(X_test)

    # Log accuracy
    accuracy=accuracy_score(y_test,y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Log classification report
    classification_rep=classification_report(y_test,y_pred,out_dict=True)
    for label, metrics in classification_rep.items():
       if isinstance(metrics,dict):
         for metric, value in metrics.items():
            mlflow.log_metric(f"{label}_{metric}",value)

    # Log confusion matrix
    conf_matrix=confusion_matrix(y_test,y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(conf_matrix,annot=True,fmt="d",cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
    plt.savefig("confusion_matrix.png")
    mlflow.log_artifcat("confusion_matrix.png")
    plt.close()

# Step 6: Test various max_features values
max_features_values=[1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
  run_expermient_tfidf_max_features(max_features)